# LeetCode 334 — Increasing Triplet Subsequence bằng Python

Link đề: https://leetcode.com/problems/increasing-triplet-subsequence/ · Độ khó: **Medium** · Chủ đề: Array, Greedy

Skeleton LeetCode:

```python
class Solution:
    def increasingTriplet(self, nums: List[int]) -> bool:
```

Ghi chú: đề, ví dụ và constraints lấy từ file đề có sẵn (bài này không phải Premium).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: có tồn tại bộ ba chỉ số `i < j < k` với `nums[i] < nums[j] < nums[k]` (**dãy con tăng ngặt** độ dài 3, **không cần liên tiếp**) hay không.
2. Đi từ **ba vòng lặp** `O(n³)` → **quy hoạch động LIS** `O(n²)` → **mảng min tiền tố / max hậu tố** `O(n)` thời gian, `O(n)` bộ nhớ → **tham lam hai biến `first`, `second`** `O(n)` thời gian, `O(1)` bộ nhớ (follow-up) → **mảng `tails` kiểu patience sorting** (tổng quát cho độ dài `k` bất kỳ).
3. Hiểu **ý nghĩa thực sự** của `first` và `second`: `first` = giá trị **nhỏ nhất** có thể kết thúc một dãy con tăng độ dài 1; `second` = giá trị **nhỏ nhất** có thể kết thúc một dãy con tăng độ dài 2 — chứ **không** nhất thiết là hai phần tử của cùng một bộ ba cuối cùng.
4. Biết vì sao dùng `<=` khi cập nhật (xử lý giá trị trùng) và vì sao dãy con **ngặt**.
5. Tránh các lỗi kinh điển: dùng `<` thay `<=` khi cập nhật, tưởng "dãy con" là "đoạn liên tiếp", reset `second` khi cập nhật `first`, sắp xếp lại mảng, và dùng `bisect_right` cho dãy con tăng ngặt.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trả về `true` nếu tồn tại bộ ba chỉ số `(i, j, k)` sao cho `i < j < k` và `nums[i] < nums[j] < nums[k]`. Nếu không tồn tại, trả về `false`.

**Ví dụ 1:** `nums = [1,2,3,4,5]` → `true` (mọi bộ ba `i < j < k` đều thỏa).

**Ví dụ 2:** `nums = [5,4,3,2,1]` → `false`.

**Ví dụ 3:** `nums = [2,1,5,0,4,6]` → `true` (bộ ba `(1, 4, 5)`: `1 < 4 < 6`).

**Ràng buộc:** `1 <= nums.length <= 5·10^5`; `-2^31 <= nums[i] <= 2^31 - 1`.

**Follow up:** làm được trong `O(n)` thời gian và `O(1)` bộ nhớ phụ không?

In [1]:
from typing import List
from bisect import bisect_left, bisect_right
import random
import time

examples = [
    ([1, 2, 3, 4, 5], True),
    ([5, 4, 3, 2, 1], False),
    ([2, 1, 5, 0, 4, 6], True),
]
for nums, expected in examples:
    print(nums, "| kỳ vọng:", expected)

[1, 2, 3, 4, 5] | kỳ vọng: True
[5, 4, 3, 2, 1] | kỳ vọng: False
[2, 1, 5, 0, 4, 6] | kỳ vọng: True


## 3. Kiến thức nền

### 3.1. Dãy con (subsequence) khác đoạn liên tiếp (subarray)

Dãy con giữ **thứ tự tương đối** nhưng có thể **bỏ qua** phần tử ở giữa. Bộ ba `(i, j, k)` với `i < j < k` không cần `j = i + 1`, `k = j + 1`. Ví dụ `[2,1,5,0,4,6]` có bộ ba `(1, 4, 5)` (giá trị `1, 4, 6`) bỏ qua `5` và `0`.
#### Code demo

In [2]:
nums = [2, 1, 5, 0, 4, 6]
i, j, k = 1, 4, 5
print([nums[i], nums[j], nums[k]], "| tăng ngặt:", nums[i] < nums[j] < nums[k], "| liên tiếp:", j == i + 1 and k == j + 1)

[1, 4, 6] | tăng ngặt: True | liên tiếp: False


### 3.2. "Tăng ngặt" nghĩa là `<` chứ không phải `<=`

`nums[i] < nums[j] < nums[k]` — hai phần tử **bằng nhau** không đủ. Vì thế `[1,1,1]`, `[1,2,2]` đều **không** có bộ ba.
#### Code demo

In [3]:
from itertools import combinations


def has_triplet_naive(nums):
    return any(nums[i] < nums[j] < nums[k] for i, j, k in combinations(range(len(nums)), 3))


print(has_triplet_naive([1, 1, 1]), has_triplet_naive([1, 2, 2]), has_triplet_naive([1, 2, 3]))

False False True


### 3.3. Dãy con tăng dài nhất (LIS) và mối liên hệ

Có bộ ba tăng ngặt ⟺ độ dài dãy con tăng ngặt dài nhất **LIS `≥ 3`**. Bài này là trường hợp đặc biệt "LIS có đạt 3 không?" — và nhờ giới hạn `3`, ta có thể giải bằng hai biến thay vì mảng `dp`.
#### Code demo

In [4]:
def lis_length(nums):
    dp = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp, default=0)


print(lis_length([2, 1, 5, 0, 4, 6]), lis_length([5, 4, 3, 2, 1]), lis_length([1, 1, 1]))

3 1 1


### 3.4. `bisect_left` và `bisect_right`

`bisect_left(a, x)`: vị trí đầu tiên mà `a[pos] >= x` (đặt `x` **trước** các phần tử bằng `x`). `bisect_right(a, x)`: vị trí đầu tiên mà `a[pos] > x`. Với dãy tăng **ngặt**, thay thế phải dùng `bisect_left` (thay phần tử `>= x` đầu tiên); `bisect_right` cho dãy **không giảm**.
#### Code demo

In [5]:
a = [1, 3, 3, 5]
print("bisect_left(a, 3) =", bisect_left(a, 3), "| bisect_right(a, 3) =", bisect_right(a, 3))

bisect_left(a, 3) = 1 | bisect_right(a, 3) = 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`[2, 1, 5, 0, 4, 6]`. Duyệt từ trái sang phải, giữ hai "ứng viên":

- `first` = giá trị nhỏ nhất đã thấy (có thể là phần tử đầu của một dãy tăng độ dài 1).
- `second` = giá trị nhỏ nhất có thể là phần tử thứ hai của một dãy tăng độ dài 2 (tức đã có phần tử nhỏ hơn đứng trước nó).

| đọc `x` | hành động | `first` | `second` |
|---|---|---|---|
| (khởi tạo) | | `+∞` | `+∞` |
| 2 | `x <= first` ⇒ `first = 2` | 2 | `+∞` |
| 1 | `x <= first` ⇒ `first = 1` | 1 | `+∞` |
| 5 | `5 > first`, `5 <= second` ⇒ `second = 5` | 1 | 5 |
| 0 | `x <= first` ⇒ `first = 0` | 0 | 5 |
| 4 | `4 > first`, `4 <= second` ⇒ `second = 4` | 0 | 4 |
| 6 | `6 > second` ⇒ **tìm thấy** (có dãy `?, 4, 6` với `? < 4` đứng trước) | | |

Khi đọc `6 > second = 4`: tồn tại một số `< 4` đứng **trước** `4` (chính là `1`, đã từng là `first`) ⇒ bộ ba `1, 4, 6`. Chú ý: lúc đó `first = 0` **không** đứng trước `4`, nhưng điều đó không quan trọng — `second` ghi nhớ rằng đã từng có một phần tử nhỏ hơn nó đứng trước.

## 5. Cách 1 — Brute Force / baseline: ba vòng lặp

**Ý tưởng.** Thử mọi bộ ba `i < j < k`, kiểm tra `nums[i] < nums[j] < nums[k]`.
#### Code demo

In [6]:
def triplet_bruteforce(nums):
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] < nums[j]:
                for k in range(j + 1, n):
                    if nums[j] < nums[k]:
                        return True
    return False


for nums, expected in examples:
    print(nums, "->", triplet_bruteforce(nums), "| kỳ vọng:", expected)

[1, 2, 3, 4, 5] -> True | kỳ vọng: True
[5, 4, 3, 2, 1] -> False | kỳ vọng: False
[2, 1, 5, 0, 4, 6] -> True | kỳ vọng: True


**Độ phức tạp (Cách 1).** **Time `O(n³)`**, **Space `O(1)`**. Với `n = 5·10^5` hoàn toàn không khả thi.

**Nút thắt.** Mỗi `j` thử lại mọi `i` và `k`; nhưng chỉ cần biết **có tồn tại** `i < j` nhỏ hơn và `k > j` lớn hơn, không cần biết chính xác là ai.

## 6. Ý tưởng thuật toán / bước đột phá

**Hai biến tham lam.**

- `first`: giá trị nhỏ nhất đã gặp.
- `second`: giá trị nhỏ nhất đã gặp **mà phía trước nó có một phần tử nhỏ hơn** (kết thúc của dãy tăng độ dài 2 nhỏ nhất).

Với mỗi `x`:

1. `x <= first` → `first = x`.
2. ngược lại nếu `x <= second` → `second = x`.
3. ngược lại (`x > second`) → **có bộ ba**, trả `True`.

Giá trị nhỏ hơn luôn "tốt hơn" cho việc mở rộng dãy sau này, nên giữ giá trị nhỏ nhất là đủ.

## 7. Các cách tiếp theo

### Cách 2 — Quy hoạch động LIS `O(n²)`

**Giải thích.** `dp[i]` = độ dài dãy con tăng ngặt dài nhất **kết thúc tại** `i`. `dp[i] = 1 + max(dp[j])` với `j < i`, `nums[j] < nums[i]`. Có bộ ba nếu `max(dp) >= 3` (có thể dừng sớm khi `dp[i] >= 3`).
#### Code demo

In [7]:
def triplet_dp(nums):
    n = len(nums)
    dp = [1] * n
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i] and dp[j] + 1 > dp[i]:
                dp[i] = dp[j] + 1
        if dp[i] >= 3:
            return True
    return False


for nums, expected in examples:
    print(nums, "->", triplet_dp(nums), "| kỳ vọng:", expected)

[1, 2, 3, 4, 5] -> True | kỳ vọng: True
[5, 4, 3, 2, 1] -> False | kỳ vọng: False
[2, 1, 5, 0, 4, 6] -> True | kỳ vọng: True


**Độ phức tạp Cách 2:** **Time `O(n²)`**, **Space `O(n)`**.

### Cách 3 — Min tiền tố và max hậu tố `O(n)` thời gian, `O(n)` bộ nhớ

**Giải thích.** Bộ ba có **phần tử giữa** `nums[j]`. Với mỗi `j`: tồn tại `i < j` có `nums[i] < nums[j]` ⟺ `prefix_min[j-1] < nums[j]`; tồn tại `k > j` có `nums[k] > nums[j]` ⟺ `suffix_max[j+1] > nums[j]`. Chỉ cần kiểm tra hai điều kiện đó.
#### Code demo

In [8]:
def triplet_prefix_suffix(nums):
    n = len(nums)
    if n < 3:
        return False
    prefix_min = [0] * n
    prefix_min[0] = nums[0]
    for i in range(1, n):
        prefix_min[i] = min(prefix_min[i - 1], nums[i])
    suffix_max = [0] * n
    suffix_max[-1] = nums[-1]
    for i in range(n - 2, -1, -1):
        suffix_max[i] = max(suffix_max[i + 1], nums[i])
    for j in range(1, n - 1):
        if prefix_min[j - 1] < nums[j] < suffix_max[j + 1]:
            return True
    return False


for nums, expected in examples:
    print(nums, "->", triplet_prefix_suffix(nums), "| kỳ vọng:", expected)

[1, 2, 3, 4, 5] -> True | kỳ vọng: True
[5, 4, 3, 2, 1] -> False | kỳ vọng: False
[2, 1, 5, 0, 4, 6] -> True | kỳ vọng: True


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** (hai mảng phụ).

### Cách 4 — Tham lam hai biến `first`, `second` (cách nộp chuẩn)

**Giải thích.** Như mục 6.
#### Code demo

In [9]:
def triplet_greedy(nums):
    first = second = float("inf")
    for x in nums:
        if x <= first:
            first = x                    # giá trị nhỏ nhất đến giờ
        elif x <= second:
            second = x                   # có số nhỏ hơn đứng trước x
        else:
            return True                  # x > second > (số đứng trước second)
    return False


for nums, expected in examples:
    print(nums, "->", triplet_greedy(nums), "| kỳ vọng:", expected)

[1, 2, 3, 4, 5] -> True | kỳ vọng: True
[5, 4, 3, 2, 1] -> False | kỳ vọng: False
[2, 1, 5, 0, 4, 6] -> True | kỳ vọng: True


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 5 — Mảng `tails` (patience sorting) dừng khi đạt độ dài 3

**Giải thích.** Giữ `tails[l]` = giá trị nhỏ nhất kết thúc một dãy tăng ngặt có độ dài `l + 1`. Với mỗi `x`, `pos = bisect_left(tails, x)`: nếu `pos == len(tails)` thì thêm `x` (kéo dài dãy); ngược lại thay `tails[pos] = x`. Có bộ ba khi `len(tails) == 3`. Đây chính là Cách 4 viết tổng quát (`first = tails[0]`, `second = tails[1]`) và dùng được cho độ dài `k` bất kỳ.
#### Code demo

In [10]:
def triplet_tails(nums, k=3):
    tails = []
    for x in nums:
        pos = bisect_left(tails, x)
        if pos == len(tails):
            tails.append(x)
            if len(tails) == k:
                return True
        else:
            tails[pos] = x
    return False


for nums, expected in examples:
    print(nums, "->", triplet_tails(nums), "| kỳ vọng:", expected)

[1, 2, 3, 4, 5] -> True | kỳ vọng: True
[5, 4, 3, 2, 1] -> False | kỳ vọng: False
[2, 1, 5, 0, 4, 6] -> True | kỳ vọng: True


**Độ phức tạp Cách 5:** **Time `O(n log k)`** với `k = 3` (nhị phân trên mảng tối đa 3 phần tử) ≈ `O(n)`, **Space `O(k) = O(1)`**.

> Cách 4 là bản nộp thông dụng và là lời giải cho follow-up. Cách 5 tổng quát hóa cho "dãy con tăng độ dài `k`".

## 8. TRACE từng bước

Ta trace Cách 4 trên ba mảng: `[2,1,5,0,4,6]` (có), `[5,4,3,2,1]` (không), và `[20,100,10,12,5,13]` (ví dụ nổi tiếng cho thấy `first` không nhất thiết thuộc bộ ba cuối cùng).
#### Code demo

In [11]:
def trace_greedy(nums):
    first = second = float("inf")
    print(f"nums = {nums}")
    for idx, x in enumerate(nums):
        if x <= first:
            first = x
            act = f"x <= first -> first = {x}"
        elif x <= second:
            second = x
            act = f"x <= second -> second = {x}"
        else:
            print(f"  idx={idx} x={x:>3}: x > second ({second}) -> TÌM THẤY bộ ba, trả True")
            return True
        print(f"  idx={idx} x={x:>3}: {act:<28} | first={first} second={second}")
    print("  hết mảng -> False")
    return False


trace_greedy([2, 1, 5, 0, 4, 6])
print()
trace_greedy([5, 4, 3, 2, 1])
print()
trace_greedy([20, 100, 10, 12, 5, 13])

nums = [2, 1, 5, 0, 4, 6]
  idx=0 x=  2: x <= first -> first = 2      | first=2 second=inf
  idx=1 x=  1: x <= first -> first = 1      | first=1 second=inf
  idx=2 x=  5: x <= second -> second = 5    | first=1 second=5
  idx=3 x=  0: x <= first -> first = 0      | first=0 second=5
  idx=4 x=  4: x <= second -> second = 4    | first=0 second=4
  idx=5 x=  6: x > second (4) -> TÌM THẤY bộ ba, trả True

nums = [5, 4, 3, 2, 1]
  idx=0 x=  5: x <= first -> first = 5      | first=5 second=inf
  idx=1 x=  4: x <= first -> first = 4      | first=4 second=inf
  idx=2 x=  3: x <= first -> first = 3      | first=3 second=inf
  idx=3 x=  2: x <= first -> first = 2      | first=2 second=inf
  idx=4 x=  1: x <= first -> first = 1      | first=1 second=inf
  hết mảng -> False

nums = [20, 100, 10, 12, 5, 13]
  idx=0 x= 20: x <= first -> first = 20     | first=20 second=inf
  idx=1 x=100: x <= second -> second = 100  | first=20 second=100
  idx=2 x= 10: x <= first -> first = 10     | first=10 second=1

True

Nhận xét với `[20,100,10,12,5,13]`: sau `5`, `first = 5` nhưng `5` đứng **sau** `12`. Bộ ba thật là `10, 12, 13`. Khi `x = 13 > second = 12`, ta kết luận đúng vì `second = 12` đã được xác nhận có một phần tử nhỏ hơn đứng **trước** nó (`10`). Biến `first` chỉ phục vụ để **tạo ra** `second` tốt hơn trong tương lai.

Bảng tóm tắt (`[2,1,5,0,4,6]`):

| `x` | `first` | `second` | kết quả |
|---|---|---|---|
| 2 | 2 | `+∞` | |
| 1 | 1 | `+∞` | |
| 5 | 1 | 5 | |
| 0 | 0 | 5 | |
| 4 | 0 | 4 | |
| 6 | | | `6 > 4` ⇒ `True` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4. Đúng `class Solution`, đúng `increasingTriplet(self, nums: List[int]) -> bool`.
#### Code demo

In [12]:
class Solution:
    def increasingTriplet(self, nums: List[int]) -> bool:
        first = second = float("inf")
        for x in nums:
            if x <= first:                 # giá trị nhỏ nhất đến giờ (kể cả bằng, để xử lý trùng)
                first = x
            elif x <= second:              # có một số nhỏ hơn x đứng trước: x là ứng viên "giữa"
                second = x
            else:                          # x > second: tồn tại a < second < x với a đứng trước second
                return True
        return False


sol = Solution()
for nums, expected in examples:
    assert sol.increasingTriplet(nums) == expected, (nums, expected)
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

Bất biến sau khi xử lý tiền tố `nums[0..t]` (chưa trả `True`):

- `first` = giá trị **nhỏ nhất** trong tiền tố (kết thúc nhỏ nhất của dãy tăng độ dài 1).
- `second` = giá trị **nhỏ nhất** `v` trong tiền tố mà **trước `v` có một phần tử nhỏ hơn `v`** (kết thúc nhỏ nhất của dãy tăng độ dài 2); `+∞` nếu chưa có.

**Chứng minh bảo toàn.** Đọc `x`:

- `x <= first`: `x` là giá trị nhỏ nhất mới ⇒ `first = x`. `second` không đổi vì dãy độ dài 2 kết thúc tại `x` không tồn tại (không có số nào `< x` đứng trước).
- `first < x <= second`: có `first < x` đứng trước `x` ⇒ `x` kết thúc một dãy độ dài 2 và nhỏ hơn hoặc bằng `second` cũ ⇒ `second = x`.
- `x > second`: có `a < second` đứng trước `second` (theo bất biến) và `second < x` ⇒ bộ ba `(a, second, x)` ⇒ **trả `True`**.

**Đủ và cần.** Nếu có bộ ba thật `(a, b, c)` thì khi đọc `c` (nếu chưa trả `True` trước đó): `second ≤ b` (vì `b` kết thúc một dãy độ dài 2 trước `c`) nên `c > b ≥ second` ⇒ trả `True`. Ngược lại, khi trả `True` thì bộ ba có thật như trên.

Vì sao dùng `<=`: với giá trị trùng, `x == first` nên cập nhật `first` (không tạo dãy độ dài 2 mới vì cần **ngặt**); `x == second` thì giữ `second` (không đủ để tăng độ dài).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (≤ `5·10^5`).

| Cách | Time | Space |
|---|---|---|
| 1. Ba vòng lặp | `O(n³)` | `O(1)` |
| 2. DP LIS | `O(n²)` | `O(n)` |
| 3. Min tiền tố + max hậu tố | `O(n)` | `O(n)` |
| 4. Tham lam hai biến | `O(n)` | `O(1)` |
| 5. `tails` (dừng ở 3) | `O(n)` (nhị phân trên ≤ 3 phần tử) | `O(1)` |

Với `n = 5·10^5`: Cách 4 cỡ `5·10^5` phép (khoảng vài chục mili-giây trong Python); Cách 2 cỡ `10^11` (không chạy được); Cách 3 `1.5·10^6` phép cùng hai mảng 4MB.

## 12. Bộ test

`oracle` là brute force ba vòng lặp (`triplet_bruteforce`), **độc lập** với tham lam. Kiểm tra ví dụ, biên, tính chất, ngẫu nhiên có seed và dữ liệu lớn.
#### Code demo

In [13]:
def oracle(nums):
    return triplet_bruteforce(nums)


impls = {
    "bruteforce": triplet_bruteforce,
    "dp": triplet_dp,
    "prefix_suffix": triplet_prefix_suffix,
    "greedy": triplet_greedy,
    "tails": triplet_tails,
    "submit": sol.increasingTriplet,
}

for nums, expected in examples:
    for key, fn in impls.items():
        assert fn(list(nums)) == expected, (key, nums, expected, fn(list(nums)))
print("PASS ví dụ của đề")

PASS ví dụ của đề


### 12.1. Các trường hợp biên
#### Code demo

In [14]:
edge_cases = [
    ([1], False, "một phần tử"),
    ([1, 2], False, "hai phần tử"),
    ([1, 2, 3], True, "đúng ba phần tử tăng"),
    ([3, 2, 1], False, "ba phần tử giảm"),
    ([1, 1, 1], False, "ba phần tử bằng nhau"),
    ([1, 2, 2], False, "tăng rồi bằng"),
    ([1, 1, 2, 2], False, "chỉ có hai giá trị phân biệt"),
    ([1, 3, 0, 4], True, "bộ ba không liên tiếp: 1, 3, 4"),
    ([20, 100, 10, 12, 5, 13], True, "10, 12, 13 (first không thuộc bộ ba)"),
    ([5, 1, 5, 5, 2, 5, 4], True, "1, 2, 4"),
    ([6, 7, 1, 2], False, "hai cặp tăng rời nhau, không nối được"),
    ([-2 ** 31, 0, 2 ** 31 - 1], True, "giá trị biên 32 bit"),
    ([2 ** 31 - 1, 0, -2 ** 31], False, "giá trị biên giảm"),
    ([1, 5, 0, 4, 1, 3], True, "có bộ ba (0, 1, 3)"),
    ([4, 5, 1, 2, 3], True, "1, 2, 3"),
]
for nums, expected, label in edge_cases:
    assert oracle(nums) == expected, (label, "oracle", oracle(nums), expected)
    for key, fn in impls.items():
        assert fn(list(nums)) == expected, (key, label, nums, expected, fn(list(nums)))
    print("PASS", label)

PASS một phần tử
PASS hai phần tử
PASS đúng ba phần tử tăng
PASS ba phần tử giảm
PASS ba phần tử bằng nhau
PASS tăng rồi bằng
PASS chỉ có hai giá trị phân biệt
PASS bộ ba không liên tiếp: 1, 3, 4
PASS 10, 12, 13 (first không thuộc bộ ba)
PASS 1, 2, 4
PASS hai cặp tăng rời nhau, không nối được
PASS giá trị biên 32 bit
PASS giá trị biên giảm
PASS có bộ ba (0, 1, 3)
PASS 1, 2, 3


### 12.2. Tính chất và phản ví dụ

- **Có bộ ba ⟺ LIS `≥ 3`** (so với `lis_length`).
- **Đối xứng**: `f(nums) == f([-x for x in reversed(nums)])` (đảo thứ tự và đổi dấu giữ nguyên tính tăng ngặt).
- **Đơn điệu theo chèn**: nếu `nums` có bộ ba thì chèn thêm phần tử bất kỳ vẫn có.
- Thêm một hằng số vào mọi phần tử không đổi kết quả.
- **Phản ví dụ** cho "dãy con = đoạn liên tiếp": `[1, 3, 0, 4]`.
#### Code demo

In [15]:
rng = random.Random(334)
for _ in range(600):
    n = rng.randint(1, 12)
    nums = [rng.randint(-5, 5) for _ in range(n)]
    r = sol.increasingTriplet(nums)
    assert r == (lis_length(nums) >= 3)
    assert r == sol.increasingTriplet([-x for x in reversed(nums)])
    assert r == sol.increasingTriplet([x + 7 for x in nums])
    if r:
        extra = nums[:]
        extra.insert(rng.randint(0, n), rng.randint(-5, 5))
        assert sol.increasingTriplet(extra)
print("PASS: LIS >= 3, đối xứng (đảo + đổi dấu), cộng hằng số, đơn điệu theo chèn")


def sai_lien_tiep(nums):
    return any(nums[i] < nums[i + 1] < nums[i + 2] for i in range(len(nums) - 2))      # SAI: chỉ xét đoạn liên tiếp


bad = [1, 3, 0, 4]
print("chỉ đoạn liên tiếp:", sai_lien_tiep(bad), "| đúng:", sol.increasingTriplet(bad))
assert sai_lien_tiep(bad) != sol.increasingTriplet(bad)

PASS: LIS >= 3, đối xứng (đảo + đổi dấu), cộng hằng số, đơn điệu theo chèn
chỉ đoạn liên tiếp: False | đúng: True


### 12.3. Kiểm tra ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [16]:
rng = random.Random(2334)
counts = [0, 0]
for trial in range(5000):
    n = rng.randint(1, 12)
    lo, hi = rng.choice([(0, 2), (0, 3), (-5, 5), (0, 20), (-2 ** 31, 2 ** 31 - 1)])
    mode = rng.random()
    if mode < 0.3:
        nums = sorted((rng.randint(lo, hi) for _ in range(n)), reverse=True)           # giảm dần: thường False
        if rng.random() < 0.5 and n >= 2:
            a, b = rng.randrange(n), rng.randrange(n)
            nums[a], nums[b] = nums[b], nums[a]
    else:
        nums = [rng.randint(lo, hi) for _ in range(n)]
    expected = oracle(nums)
    counts[expected] += 1
    for key, fn in impls.items():
        assert fn(list(nums)) == expected, (key, nums, expected, fn(list(nums)))
print(f"PASS 5000 test ngẫu nhiên với oracle ba vòng lặp (True: {counts[1]}, False: {counts[0]})")
assert counts[0] > 300 and counts[1] > 300

# n = 5 * 10^5
N = 500_000
rng = random.Random(7)
desc = list(range(N, 0, -1))
big_cases = [
    ("giảm dần (False)", desc, False),
    ("toàn bằng nhau (False)", [7] * N, False),
    ("tăng dần (True, tìm thấy ngay)", list(range(N)), True),
    ("giảm dần rồi 3 phần tử tăng ở cuối (True)", [x + 10 for x in desc[:-3]] + [1, 2, 3], True),
    ("hai giá trị xen kẽ (False)", [1, 2] * (N // 2), False),
    ("ngẫu nhiên 32 bit", [rng.randint(-2 ** 31, 2 ** 31 - 1) for _ in range(N)], True),
    ("hai dãy giảm rời nhau (False)", list(range(N // 2, 0, -1)) + list(range(N // 2, 0, -1)), False),
]
for label, nums, expected in big_cases:
    row, answers = [], set()
    for key in ("prefix_suffix", "greedy", "tails", "submit"):
        t0 = time.perf_counter()
        r = impls[key](nums)
        dt = (time.perf_counter() - t0) * 1000
        answers.add(r)
        row.append(f"{key}={dt:7.1f}ms")
    assert answers == {expected}, (label, answers, expected)
    print(f"{label:<44} | " + " ".join(row))
print("PASS n = 5*10^5")

PASS 5000 test ngẫu nhiên với oracle ba vòng lặp (True: 2246, False: 2754)


giảm dần (False)                             | prefix_suffix=   79.8ms greedy=    8.9ms tails=   23.3ms submit=    8.9ms
toàn bằng nhau (False)                       | prefix_suffix=   98.0ms greedy=    9.4ms tails=   25.0ms submit=    9.2ms


tăng dần (True, tìm thấy ngay)               | prefix_suffix=   71.6ms greedy=    0.0ms tails=    0.0ms submit=    0.0ms
giảm dần rồi 3 phần tử tăng ở cuối (True)    | prefix_suffix=   98.6ms greedy=    9.6ms tails=   22.3ms submit=    9.3ms


hai giá trị xen kẽ (False)                   | prefix_suffix=   92.6ms greedy=   11.6ms tails=   23.5ms submit=   11.9ms
ngẫu nhiên 32 bit                            | prefix_suffix=   58.2ms greedy=    0.0ms tails=    0.0ms submit=    0.0ms
hai dãy giảm rời nhau (False)                | prefix_suffix=   80.1ms greedy=   11.9ms tails=   23.5ms submit=   11.4ms
PASS n = 5*10^5


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `<` thay `<=` khi cập nhật (bỏ qua giá trị trùng)
#### Code sai

In [17]:
def sai_nghiem_ngat_khi_cap_nhat(nums):
    first = second = float("inf")
    for x in nums:
        if x < first:                         # SAI: phải là x <= first
            first = x
        elif x < second:                      # SAI: phải là x <= second
            second = x
        else:
            return True
    return False


for nums in ([1, 1, 1], [1, 2, 2]):
    print(nums, "sai:", sai_nghiem_ngat_khi_cap_nhat(nums), "| đúng:", sol.increasingTriplet(nums))
assert sai_nghiem_ngat_khi_cap_nhat([1, 1, 1]) != sol.increasingTriplet([1, 1, 1])

[1, 1, 1] sai: True | đúng: False
[1, 2, 2] sai: True | đúng: False


**Vì sao sai:** với `x == first`, không được đi xuống nhánh `second`/`return True`: giá trị bằng nhau **không** tăng ngặt. Với `[1,1,1]`: `first = 1`, rồi `x = 1` không `< first` nên rơi vào `second = 1`, rồi `x = 1` không `< second` nên trả `True` (sai). **Code đúng:** `x <= first`, `x <= second`.

### Lỗi 2: Nhầm dãy con với đoạn liên tiếp
Xem `sai_lien_tiep` ở mục 12.2: `[1, 3, 0, 4]` cho `False` nhưng đáp án là `True` (bộ ba `1, 3, 4`). **Vì sao sai:** đề không yêu cầu liên tiếp. **Code đúng:** tham lam/min–max/`tails`.

### Lỗi 3: Reset `second` mỗi khi cập nhật `first`
#### Code sai

In [18]:
def sai_reset_second(nums):
    first = second = float("inf")
    for x in nums:
        if x <= first:
            first = x
            second = float("inf")             # SAI: làm mất thông tin dãy độ dài 2 đã tìm thấy
        elif x <= second:
            second = x
        else:
            return True
    return False


bad = [1, 3, 0, 4]
print("sai:", sai_reset_second(bad), "| đúng:", sol.increasingTriplet(bad))
assert sai_reset_second(bad) != sol.increasingTriplet(bad)

sai: False | đúng: True


**Vì sao sai:** `second` ghi nhớ rằng **đã từng** có dãy tăng độ dài 2 kết thúc ở giá trị đó, độc lập với `first` hiện tại. Khi `first` bị hạ xuống `0` (sau `3`), dãy `1, 3` vẫn là bằng chứng hợp lệ; reset `second` làm mất nó và bỏ lỡ `4`. **Code đúng:** không đụng `second` khi cập nhật `first`.

### Lỗi 4: Sắp xếp mảng trước
#### Code sai

In [19]:
def sai_sap_xep(nums):
    s = sorted(set(nums))                     # SAI: phá thứ tự vị trí
    return len(s) >= 3


print("[5,4,3,2,1] sai:", sai_sap_xep([5, 4, 3, 2, 1]), "| đúng:", sol.increasingTriplet([5, 4, 3, 2, 1]))
assert sai_sap_xep([5, 4, 3, 2, 1]) != sol.increasingTriplet([5, 4, 3, 2, 1])

[5,4,3,2,1] sai: True | đúng: False


**Vì sao sai:** bài hỏi về **thứ tự vị trí** (`i < j < k`); sắp xếp làm mất thứ tự gốc. Mảng giảm dần có `5` giá trị phân biệt nhưng **không** có bộ ba tăng. **Code đúng:** giữ nguyên thứ tự, duyệt một lượt.

### Lỗi 5: Dùng `bisect_right` cho dãy tăng ngặt
#### Code sai

In [20]:
def sai_bisect_right(nums, k=3):
    tails = []
    for x in nums:
        pos = bisect_right(tails, x)          # SAI: cho dãy KHÔNG GIẢM; ngặt cần bisect_left
        if pos == len(tails):
            tails.append(x)
            if len(tails) == k:
                return True
        else:
            tails[pos] = x
    return False


print("[1,1,1] sai:", sai_bisect_right([1, 1, 1]), "| đúng:", sol.increasingTriplet([1, 1, 1]))
assert sai_bisect_right([1, 1, 1]) != sol.increasingTriplet([1, 1, 1])

[1,1,1] sai: True | đúng: False


**Vì sao sai:** `bisect_right` cho phép phần tử bằng nhau nối tiếp nhau (`[1,1,1]` thành "dãy tăng độ dài 3"). Đề yêu cầu **ngặt**. **Code đúng:** `bisect_left`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**dãy con tăng độ dài nhỏ cố định**" (3) và yêu cầu `O(1)` bộ nhớ → **hai biến tham lam** `first`, `second`.
- Tổng quát: **`tails` + `bisect_left`** (patience sorting) cho độ dài `k` bất kỳ; LIS độ dài tối đa = `len(tails)` ở cuối.
- Nhớ ý nghĩa: `first` = kết thúc nhỏ nhất của dãy độ dài 1; `second` = kết thúc nhỏ nhất của dãy độ dài 2.
- Dùng `<=` khi cập nhật để xử lý trùng (tăng **ngặt**).
- Phương án dự phòng dễ chứng minh: **min tiền tố** + **max hậu tố** cho phần tử giữa.
- Họ hàng: **Longest Increasing Subsequence** (300), **Number of LIS** (673), **Russian Doll Envelopes** (354), **Increasing Subsequences** (491), **132 Pattern** (456).
- Thử: `[1,1,1]`, `[5,4,3,2,1]`, `[1,3,0,4]`, `[20,100,10,12,5,13]`, `n = 5·10^5`.

## 15. Biến thể của bài

1. **Dãy con tăng độ dài `k`** bất kỳ: `tails` + `bisect_left`, dừng khi `len(tails) == k`.
2. **Dãy con tăng **không giảm** độ dài 3** (cho phép bằng nhau): đổi `bisect_left` → `bisect_right` / đổi `<=` thành `<` ở các nhánh (xem Lỗi 1).
3. **Trả về một bộ ba** `(i, j, k)` cụ thể (cần lưu chỉ số và `prev`).
4. **132 Pattern** (456): `i < j < k` với `nums[i] < nums[k] < nums[j]`.
5. **Đếm số bộ ba tăng** (Fenwick/DP).
#### Code demo

In [21]:
def has_increasing_k(nums, k):
    return triplet_tails(nums, k)


def find_triplet_indices(nums):
    """Trả (i, j, k) của một bộ ba tăng ngặt, hoặc None. O(n)."""
    n = len(nums)
    left_min_idx = [0] * n                                   # chỉ số của min tiền tố
    for t in range(1, n):
        left_min_idx[t] = t if nums[t] < nums[left_min_idx[t - 1]] else left_min_idx[t - 1]
    right_max_idx = [n - 1] * n
    for t in range(n - 2, -1, -1):
        right_max_idx[t] = t if nums[t] > nums[right_max_idx[t + 1]] else right_max_idx[t + 1]
    for j in range(1, n - 1):
        i, k = left_min_idx[j - 1], right_max_idx[j + 1]
        if nums[i] < nums[j] < nums[k]:
            return i, j, k
    return None


def count_increasing_triplets(nums):
    """Đếm số bộ ba tăng ngặt, O(n^2) đơn giản (mỗi j đếm số nhỏ hơn bên trái và lớn hơn bên phải)."""
    total = 0
    for j in range(len(nums)):
        less = sum(1 for i in range(j) if nums[i] < nums[j])
        more = sum(1 for k in range(j + 1, len(nums)) if nums[k] > nums[j])
        total += less * more
    return total


print(has_increasing_k([1, 2, 3, 4, 5], 5), has_increasing_k([1, 2, 3, 4, 5], 6), has_increasing_k([5, 1, 6, 2, 7, 3, 8], 4))
print(find_triplet_indices([2, 1, 5, 0, 4, 6]), find_triplet_indices([5, 4, 3, 2, 1]))
print(count_increasing_triplets([1, 2, 3, 4]), count_increasing_triplets([1, 1, 1]))   # 4 và 0

True False True
(1, 2, 5) None
4 0


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ba vòng lặp | thử mọi `(i, j, k)` | `O(n³)` | `O(1)` | oracle `n ≤ 12` |
| 2. DP LIS | `dp[i]` = LIS kết thúc tại `i` | `O(n²)` | `O(n)` | tổng quát, `n` nhỏ |
| 3. Min tiền tố + max hậu tố | phần tử giữa `j` | `O(n)` | `O(n)` | dễ chứng minh |
| 4. Hai biến tham lam | `first`, `second` | `O(n)` | `O(1)` | **bản nộp chuẩn / follow-up** |
| 5. `tails` + `bisect_left` | patience sorting, dừng ở 3 | `O(n)` | `O(1)` | độ dài `k` bất kỳ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `increasingTriplet` cho: (a) `[1,2,3]`; (b) `[1,1,2,2,3]`; (c) `[3,2,1,2,3]`; (d) `[1,2,1,2,1,2]`.

**Bài 2 (tính bằng tay).** Chạy Cách 4 cho `[5, 1, 5, 5, 2, 5, 4]`: ghi `first`, `second` sau mỗi phần tử và nói dừng ở đâu.

**Bài 3 (điền TODO).** Hoàn thiện `triplet_todo(nums)` theo Cách 4.
#### Code demo

In [22]:
def triplet_todo(nums):
    # TODO 1: first = second = float("inf")
    # TODO 2: nếu x <= first thì first = x
    # TODO 3: ngược lại nếu x <= second thì second = x
    # TODO 4: ngược lại return True; cuối cùng return False
    pass


# Tự làm xong rồi mới mở test:
# for nums, expected in examples: assert triplet_todo(nums) == expected

**Bài 4 (tự viết lại).** Viết `longest_increasing_at_most(nums, cap)` trả `min(LIS, cap)` bằng `tails` và `bisect_left` (dừng sớm khi đạt `cap`).

**Bài 5 (biến thể / mini challenge).** Viết `has_nondecreasing_triplet(nums)`: tồn tại `i < j < k` với `nums[i] <= nums[j] <= nums[k]` (cho phép bằng nhau). Thử `[1,1,1]` → `True`, `[3,2,1]` → `False`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: cần `i < j < k` **ngặt** về giá trị. Bài 2: `first` là nhỏ nhất, `second` là nhỏ nhất có số nhỏ hơn đứng trước. Bài 4: `tails` giữ giá trị kết thúc nhỏ nhất theo độ dài. Bài 5: bất đẳng thức không ngặt.
- **Mức 2 (vừa):** Bài 1: `True`, `True` (`1,2,3`), `True` (`1,2,3` ở cuối), `False` (chỉ có hai giá trị phân biệt). Bài 2: dừng ở phần tử `5` thứ ba (chỉ số 5) khi `first=1, second=2`, trả `True`. Bài 3: khung Cách 4. Bài 4: trả `len(tails)` hoặc `cap`.
- **Mức 3 (sâu):** Bài 4: nếu `len(tails) == cap` sau khi thêm thì trả `cap` ngay. Bài 5: dùng `bisect_right` thay `bisect_left` (hoặc đổi `x <= first` thành `x < first` và `x <= second` thành `x < second`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [23]:
# Bài 1
assert [sol.increasingTriplet(v) for v in ([1, 2, 3], [1, 1, 2, 2, 3], [3, 2, 1, 2, 3], [1, 2, 1, 2, 1, 2])] == [True, True, True, False]

# Bài 2
assert sol.increasingTriplet([5, 1, 5, 5, 2, 5, 4]) is True


# Bài 3
def triplet_todo_ans(nums):
    first = second = float("inf")
    for x in nums:
        if x <= first:
            first = x
        elif x <= second:
            second = x
        else:
            return True
    return False


for nums, expected in examples:
    assert triplet_todo_ans(nums) == expected


# Bài 4
def longest_increasing_at_most(nums, cap):
    tails = []
    for x in nums:
        pos = bisect_left(tails, x)
        if pos == len(tails):
            tails.append(x)
            if len(tails) == cap:
                return cap
        else:
            tails[pos] = x
    return len(tails)


rng = random.Random(41)
for _ in range(300):
    nums = [rng.randint(-5, 5) for _ in range(rng.randint(0, 12))]
    assert longest_increasing_at_most(nums, 3) == min(lis_length(nums), 3)
    assert longest_increasing_at_most(nums, 100) == lis_length(nums)


# Bài 5
def has_nondecreasing_triplet(nums):
    tails = []
    for x in nums:
        pos = bisect_right(tails, x)
        if pos == len(tails):
            tails.append(x)
            if len(tails) == 3:
                return True
        else:
            tails[pos] = x
    return False


def nondecreasing_oracle(nums):
    return any(nums[i] <= nums[j] <= nums[k] for i, j, k in combinations(range(len(nums)), 3))


assert has_nondecreasing_triplet([1, 1, 1]) is True and has_nondecreasing_triplet([3, 2, 1]) is False
for _ in range(400):
    nums = [rng.randint(-3, 3) for _ in range(rng.randint(0, 10))]
    assert has_nondecreasing_triplet(nums) == nondecreasing_oracle(nums), nums
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai biến tham lam)

```python
class Solution:
    def increasingTriplet(self, nums: List[int]) -> bool:
        first = second = float("inf")
        for x in nums:
            if x <= first:
                first = x
            elif x <= second:
                second = x
            else:
                return True
        return False
```

**Checklist**

- [ ] Hiểu `first`/`second` là **kết thúc nhỏ nhất** của dãy tăng độ dài 1/2 (không phải bộ ba cuối)?
- [ ] Dùng `<=` ở cả hai nhánh cập nhật (xử lý trùng, **ngặt**)?
- [ ] **Không** reset `second` khi cập nhật `first`?
- [ ] Không sắp xếp lại mảng; giữ thứ tự vị trí?
- [ ] Nhánh `x > second` trả `True` ngay?
- [ ] Đã thử `[1,1,1]`, `[5,4,3,2,1]`, `[1,3,0,4]`, `[20,100,10,12,5,13]`, `n = 5·10^5`?
- [ ] Skeleton đúng `class Solution` / `def increasingTriplet(self, nums: List[int]) -> bool`?